<a href="https://colab.research.google.com/github/MuTuOz/CustomMouseDriver/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuTuOz/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane 2. I picked Lane 2 because the prioritization problem is measurable in this data. Impressions are heavily concentrated, so with a review capacity of roughly 20 pages a month against 30,000 pages, which ones you pick is most of the outcome. The existing signal doesn't help with that: trend_direction == "down" is true for 54% of pages, so it flags the majority of the inventory and separates almost nothing. Lane 2 produces a ranked queue with reason codes, which is exactly what's missing. I'll develop on the 30k starter slice for speed and re-check the result on the full warehouse with client holdout.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

My work improves which pages enter this cycle's review queue, and in what order. A content strategist will act on it with 20 pages a month capacity. The two errors don't cost the same. If I rank a healthy page too high, an editor spends a few hours on it and finds little to fix. If I miss a declining page that lots of people see, nobody looks at it for another cycle. The sizes are far apart: the median top-decile page has about 23,000 impressions over 90 days versus about 81 for the median page in the bottom half. And the existing flag can't tell these apart. So a wrong call at the top of the queue is much more expensive than a wrong call anywhere else which is why I care about precision in the top K rather than accuracy over all 30,000 pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").drop_duplicates("content_id")
print(f"rows: {len(df):,} | clients: {df.client_id.nunique()} | columns: {df.shape[1]}")

# --- 1. How concentrated is visibility? ---
imp = np.sort(df.impressions_90d.values)[::-1]
total = imp.sum()
for p in (0.01, 0.10, 0.50):
    k = int(len(imp) * p)
    print(f"top {p:>4.0%} of pages ({k:>5,}) hold {imp[:k].sum()/total:>5.1%} of impressions")

x = np.sort(df.impressions_90d.values)
n = len(x)
gini = (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())
print(f"Gini (impressions_90d): {gini:.3f}")

# --- 2. How much does the existing flag separate? ---
down = df.trend_direction == "down"
print(f"\npages flagged 'down': {down.sum():,} ({down.mean():.1%} of inventory)")

# --- 3. Can the flag find the pages that matter? ---
q90 = df.impressions_90d.quantile(0.90)
down_and_big = df[down & (df.impressions_90d >= q90)]
print(f"of those, in the top impressions decile: {len(down_and_big):,} ({len(down_and_big)/down.sum():.1%} of the flag)")
print(f"...but they hold {down_and_big.impressions_90d.sum()/total:.1%} of all impressions")

rows: 30,000 | clients: 32 | columns: 44
top   1% of pages (  300) hold 24.9% of impressions
top  10% of pages (3,000) hold 70.2% of impressions
top  50% of pages (15,000) hold 98.3% of impressions
Gini (impressions_90d): 0.818

pages flagged 'down': 16,262 (54.2% of inventory)
of those, in the top impressions decile: 1,548 (9.5% of the flag)
...but they hold 33.4% of all impressions


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This work can say what it observed and no more. It can report which signals are associated with pages currently flagged as declining, produce a directional ranking of which pages a reviewer should look at first, and show whether that ranking beats a transparent rule baseline on the top 20 under client-holdout validation. The output is decision support, a queue with reason codes that a content strategist can read, question, and overrule. It cannot say that refreshing a page causes recovery, the data is observational and pages are not selected for refresh at random, so there is no counterfactual to compare against. It cannot say anything about how someone else ranks; I observe outputs like position and impressions, never inputs. And it cannot claim that a flagged page will decline or that these results hold beyond 32 clients on a prefiltered slice.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.